# Chapter 11: Mission Infeasible

**Colab slideshow notebook**  
Based on Chapter 11 of the book "Computational Thinking for Life Scientists"
by Benny Chor and Amir Rubinstein.

## From impossible to infeasible

- **Chapter 10:** problems that **no** program can solve.
- **Chapter 11:** problems that **can** be solved — but every known program takes so long that by the time it finishes,
  the answer is no longer relevant.

Knowing whether a problem has an **efficient** solution tells us when an exact answer is within reach —
and when we should settle for an approximation or a heuristic.

## Chapter goals

By the end of this chapter, we want to understand:

- polynomial vs. exponential running time,
- graph (and map) **coloring**: 2 colors is easy, 3 colors is (probably) hard,
- why coloring matters: **scheduling** and **frequency allocation**,
- **verifying** a solution vs. **finding** one — the **P vs. NP** question,
- what we do in practice with hard problems: heuristics and approximations.

## 11.1 Measuring computational complexity — a reminder

We measure the running time `T(n)` as a function of the **input size** `n`, in the **worst case**.

- **Polynomial time:** `T(n) ≤ a·n^c + b` for some constants — linear (`c = 1`), quadratic (`c = 2`), cubic (`c = 3`)…
  e.g. GC content (linear), BFS (linear in `n + m`), schoolbook multiplication (quadratic).
- **Exponential time:** e.g. `T(n) = 2ⁿ` or `T(n) = n!`.

For large inputs, polynomial algorithms win — by far.

In [ ]:
# Code snippet: how long would it take, at 10^9 simple steps per second?
import math

def human_time(steps):
    seconds = steps / 1e9
    for unit, size in [("years", 3.156e7), ("days", 86400), ("hours", 3600), ("minutes", 60)]:
        if seconds >= size:
            return f"{seconds / size:.3g} {unit}"
    return f"{seconds:.3g} seconds"

print(f"{'n':>5} | {'n^2':>16} | {'n^3':>16} | {'2^n':>16} | {'n!':>16}")
for n in [10, 20, 50, 100]:
    print(f"{n:>5} | {human_time(n**2):>16} | {human_time(n**3):>16} | "
          f"{human_time(2**n):>16} | {human_time(math.factorial(n)):>16}")

## Example: are all the elements of a list distinct?

- **Naïve:** compare every pair → `(n−1) + (n−2) + … + 1 = n(n−1)/2 = O(n²)` comparisons.
- **Sort first:** after sorting, equal elements are neighbours → `O(n log n) + O(n) = O(n log n)`.
- **A set** (hashing, Chapter 3): about `O(n)` on average.

The same problem can have algorithms of very different complexity — a slow algorithm does not mean a hard problem.

In [ ]:
# Code snippet: three algorithms for the distinctness problem
import random, time

def all_distinct_naive(lst):
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j]:
                return False
    return True

def all_distinct_sort(lst):
    s = sorted(lst)
    for i in range(len(s) - 1):
        if s[i] == s[i + 1]:
            return False
    return True

def all_distinct_set(lst):
    return len(set(lst)) == len(lst)

lst = random.sample(range(10**9), 3000)      # 3000 distinct numbers (the worst case)
for f in [all_distinct_naive, all_distinct_sort, all_distinct_set]:
    start = time.time()
    result = f(lst)
    print(f"{f.__name__:20} -> {result}   {time.time() - start:.4f} seconds")

## 11.2 Maps and coloring

Color the regions of a map so that regions with a **common border** get **different** colors — a **legal coloring**.
(A single common point is not a border.)

- A map is **k-colorable** if it has a legal coloring with `k` colors.
- **Decision problem:** given a map, is it 2-colorable? (answer: yes / no)

## 11.3 From maps to graphs

Keep only "who borders whom": one **node** per region, an **edge** between neighbouring regions.
A **coloring of a graph**: a color per node; **legal** if every edge connects two different colors.

In [ ]:
# Code snippet: graphs as adjacency matrices, and a legal-coloring check
import itertools

def graph_from_edges(n, edges):
    G = [[0] * n for _ in range(n)]
    for a, b in edges:
        G[a][b] = G[b][a] = 1
    return G

def is_legal_coloring(G, coloring):
    ''' coloring[i] = color of node i. Legal iff no edge connects two equal colors. '''
    n = len(G)
    return all(coloring[i] != coloring[j] for i in range(n) for j in range(i + 1, n) if G[i][j])

def cycle(n):
    return graph_from_edges(n, [(i, (i + 1) % n) for i in range(n)])

print("cycle of 4, colors 0,1,0,1:", is_legal_coloring(cycle(4), [0, 1, 0, 1]))
print("cycle of 4, colors 0,1,1,0:", is_legal_coloring(cycle(4), [0, 1, 1, 0]))

## Brute force: try all colorings

With `k` colors and `n` nodes there are `kⁿ` colorings. Try them all:

- if one is legal → "yes";
- if none is → "no" — and to say "no" we **must** try them all.

For `k = 2`: `2ⁿ` colorings. `n = 20` is fine; `n = 100` means `2¹⁰⁰ ≈ 1.27·10³⁰` colorings — not before the sun burns out.

In [ ]:
# Code snippet: brute-force k-colorability

def is_k_colorable_brute(G, k):
    ''' try all k^n colorings; return (a legal coloring or None, number of colorings tried) '''
    tried = 0
    for coloring in itertools.product(range(k), repeat=len(G)):
        tried += 1
        if is_legal_coloring(G, coloring):
            return coloring, tried
    return None, tried

for n in [6, 7]:
    coloring, tried = is_k_colorable_brute(cycle(n), 2)
    print(f"cycle of {n}, 2 colors: {coloring}  (tried {tried} of {2**n} colorings)")

## 2-coloring can be done efficiently

1. Color any node green.
2. Repeatedly: take a recently colored node, color all its uncolored neighbours with the **other** color.
3. If two neighbours ever get the same color → **not** 2-colorable.

This is exactly a BFS (Chapter 6) — polynomial time, no `2ⁿ`!
(And a graph is 2-colorable ⇔ it is **bipartite** ⇔ it has no odd cycle — Exercise 2 of Chapter 5.)

In [ ]:
# Code snippet: efficient 2-coloring with BFS

def two_coloring(G):
    ''' return a legal 2-coloring (list of 0/1), or None if G is not 2-colorable '''
    n = len(G)
    color = [None] * n
    for start in range(n):                 # handles graphs that are not connected
        if color[start] is not None:
            continue
        color[start] = 0
        Q = [start]
        while Q:
            u = Q.pop(0)
            for v in range(n):
                if G[u][v]:
                    if color[v] is None:
                        color[v] = 1 - color[u]     # the other color
                        Q.append(v)
                    elif color[v] == color[u]:
                        return None                 # conflict
    return color

print("cycle of 6:  ", two_coloring(cycle(6)))
print("cycle of 7:  ", two_coloring(cycle(7)))
print("cycle of 1000 (even):", "2-colorable" if two_coloring(cycle(1000)) else "not 2-colorable")

## What about 3 colors?

- Brute force: `3ⁿ` colorings — much worse than `2ⁿ`.
- The BFS idea breaks: a neighbour of a colored node now has **two** possible colors.
  The choices multiply — a **combinatorial explosion**.
- **Nobody knows a polynomial-time algorithm for 3-colorability.** Most computer scientists believe none exists —
  but there is no proof.

Watch the brute force grow: on a graph that is **not** 3-colorable (a 4-clique), it must try everything.

In [ ]:
# Code snippet: brute-force 3-coloring on graphs that are NOT 3-colorable — time grows x3 per node
import time

for n in range(6, 12):
    G = graph_from_edges(n, [(a, b) for a in range(4) for b in range(a + 1, 4)])   # a 4-clique + isolated nodes
    start = time.time()
    coloring, tried = is_k_colorable_brute(G, 3)
    print(f"n = {n:2}: tried {tried:7,} colorings in {time.time() - start:.3f} seconds")

## The four color theorem

- **1852:** Francis Guthrie, a 21-year-old student, coloring the counties of England, conjectured that
  **every** planar map can be colored with **four** colors.
- Many wrong proofs followed.
- **1976:** Appel and Haken proved it — by reducing it to 1936 specific maps, each checked **by a computer program**.
  Controversial at the time (what if the program has a bug?); later verified with theorem-proving software.
- So for planar **maps**, "4-colorable?" is always "yes".

But for general **graphs**, k-colorability for any `k > 2` has no known efficient algorithm — despite 40+ years of research.
(Not every graph comes from a map: e.g. the 5-clique cannot be drawn as a planar map.)

## 11.4 Application: scheduling

Seven courses must be scheduled on Monday, in 2-hour slots: 8–10, 10–12, 12–14.

- Node = a course; edge = the two courses share students, so they **cannot** be at the same time.
- Time slot = color. "Can we finish by 14:00?" = "is the graph **3-colorable**?"

The constraints below are an **example**.

In [ ]:
# Code snippet: scheduling as graph coloring (example constraints)
import networkx as nx
import matplotlib.pyplot as plt

courses = ["MATH", "PROGRAMMING", "BIOLOGY", "CHEMISTRY", "ART", "PHYSICS", "HISTORY"]
conflicts = [("MATH", "PROGRAMMING"), ("MATH", "CHEMISTRY"), ("MATH", "PHYSICS"),
             ("PROGRAMMING", "BIOLOGY"), ("PROGRAMMING", "ART"), ("BIOLOGY", "CHEMISTRY"),
             ("BIOLOGY", "ART"), ("CHEMISTRY", "PHYSICS"), ("PHYSICS", "HISTORY"), ("ART", "HISTORY")]
idx = {c: i for i, c in enumerate(courses)}
S = graph_from_edges(len(courses), [(idx[a], idx[b]) for a, b in conflicts])

slots = ["8-10", "10-12", "12-14"]
coloring, tried = is_k_colorable_brute(S, 3)
for c in courses:
    print(f"{c:12} -> {slots[coloring[idx[c]]]}")

g = nx.Graph([(idx[a], idx[b]) for a, b in conflicts])
nx.draw(g, nx.spring_layout(g, seed=3), labels={i: c for i, c in enumerate(courses)}, with_labels=True,
        node_color=[["tomato", "cornflowerblue", "lightgreen"][coloring[i]] for i in g.nodes],
        node_size=2200, font_size=8)
plt.show()

## Reductions

Scheduling **reduces** to graph coloring: a solution to coloring immediately gives a solution to scheduling.

- This does not make either problem easy — it shows their hardness is **tightly related**.
- An efficient algorithm for one would give an efficient algorithm for the other.

## Application: cell-phone frequencies

Antennas whose coverage areas overlap need **different** frequencies, and every frequency costs money.
Node = antenna, edge = overlapping coverage, color = frequency → the **minimum** number of colors.

## Exercise 2

Seven antennas A–G. Suppose these pairs overlap (example data):
A–B, B–C, B–D, C–E, D–E, E–F, E–G, F–G, C–F.

1. Draw the graph on paper.
2. Can it be colored with 2 colors? 3? 4? Use `is_k_colorable_brute` to check your answers.
3. How much money does the minimal coloring save, at one million dollars per frequency per year?

In [ ]:
# Exercise 2: your solution here
antennas = ["A", "B", "C", "D", "E", "F", "G"]
overlaps = [("A","B"), ("B","C"), ("B","D"), ("C","E"), ("D","E"), ("E","F"), ("E","G"), ("F","G"), ("C","F")]

## 11.5 Verifying a solution is easy

Someone hands you a coloring and claims it is a legal k-coloring.

- Count the colors (≤ k?) and check every edge — **polynomial time** (`is_legal_coloring`).
- So: we do not know how to **find** a 3-coloring efficiently, but we can **verify** one efficiently.

Not everything is easy to verify: we know no efficient way to verify that a graph is **not** k-colorable.

## P vs. NP

- **P:** problems that can be **solved** efficiently (in polynomial time).
- **NP:** problems whose solutions can be **verified** efficiently.
- **P vs. NP:** can every problem that is efficiently verifiable also be efficiently solved?

Open since the early 1970s. The Clay Mathematics Institute offers **USD 1,000,000** for a correct answer.

## Euler vs. Hamilton, again (Chapter 5)

- **Eulerian path** (every **edge** once): Euler's degree condition — efficient → in **P**.
- **Hamiltonian path** (every **node** once): easy to **verify** → in **NP**;
  no efficient algorithm is known, and most believe there is none.

Brute force tries all `n!` orders of the nodes:

In [ ]:
# Code snippet: Hamiltonian path — brute force over all n! orders, and a fast verifier

def is_hamiltonian_path(G, path):
    ''' verify: every node exactly once, consecutive nodes connected — polynomial time '''
    return sorted(path) == list(range(len(G))) and all(G[path[k]][path[k+1]] for k in range(len(path) - 1))

def hamiltonian_path_brute(G):
    tried = 0
    for order in itertools.permutations(range(len(G))):
        tried += 1
        if is_hamiltonian_path(G, order):
            return order, tried
    return None, tried

H = graph_from_edges(8, [(3,0), (0,6), (6,1), (1,7), (7,2), (2,5), (5,4), (0,1), (2,4), (3,5)])
path, tried = hamiltonian_path_brute(H)
print("Hamiltonian path:", path, f" (tried {tried:,} of {math.factorial(8):,} orders)")
print("verify (3, 0, 6, 1, 7, 2, 5, 4):", is_hamiltonian_path(H, (3, 0, 6, 1, 7, 2, 5, 4)))
print("verify (0, 1, 2, 3, 4, 5, 6, 7):", is_hamiltonian_path(H, (0, 1, 2, 3, 4, 5, 6, 7)))
print("number of orders for 20 nodes:", f"{math.factorial(20):,}")

## A biological example: the most parsimonious tree

Maximum parsimony (Chapter 5): find the evolutionary tree that needs the **fewest mutations**.

- For a **given** tree, counting the mutations is easy.
- The hard part is **choosing the tree**: the number of possible (unrooted, binary) trees on `n` species is `(2n−5)!!`.
- The problem is NP-hard. In practice: **heuristics** that find good — not guaranteed best — trees.

In [ ]:
# Code snippet: the number of possible unrooted binary trees, (2n-5)!! = 1 * 3 * 5 * ... * (2n-5)

def num_trees(n):
    result = 1
    for k in range(1, 2 * n - 4, 2):
        result *= k
    return result

for n in [5, 10, 20, 50]:
    print(f"{n:3} species: {num_trees(n):.3g} possible trees")

## 11.6 What do we do with hard problems in practice?

- **Smart exponential algorithms** — often fine for moderate input sizes
- **Heuristics** — good answers for many (not all) inputs
- **Approximation algorithms** — guaranteed "close enough" (e.g. 5 colors when 3 would suffice)

Example: **greedy coloring** — go over the nodes, give each the smallest color not used by its neighbours.
Fast — but the number of colors depends on the order:

In [ ]:
# Code snippet: greedy coloring — fast, but not always optimal

def greedy_coloring(G, order):
    color = {}
    for v in order:
        used = {color[u] for u in color if G[u][v]}
        c = 0
        while c in used:
            c += 1
        color[v] = c
    return [color[v] for v in range(len(G))]

# a "crown" graph: two rows 0..3 and 4..7; i is connected to every 4+j with j != i  (2-colorable!)
crown = graph_from_edges(8, [(i, 4 + j) for i in range(4) for j in range(4) if i != j])

good = greedy_coloring(crown, [0, 1, 2, 3, 4, 5, 6, 7])
bad = greedy_coloring(crown, [0, 4, 1, 5, 2, 6, 3, 7])
print("good order:", good, "->", len(set(good)), "colors")
print("bad order: ", bad, "->", len(set(bad)), "colors")
print("optimal (BFS 2-coloring):", two_coloring(crown))

## And AI tools?

- An AI tool is a program — it does not turn an exponential problem into a polynomial one.
- It can be an excellent **heuristic**: AlphaFold gives useful protein structures for many real proteins,
  without solving the hard problem in the worst case — and reports confidence scores you must check.
- Because NP problems are **easy to verify**, a proposed answer (from a person, a heuristic, or an AI) can be **checked** efficiently —
  `is_legal_coloring`, `is_hamiltonian_path`. Verify before you trust.

**AI moves the line of what is practical — not the line of what is possible.**

## Chapter 11 Summary

In this chapter we studied:

- polynomial vs. exponential running time
- map and graph coloring: 2-coloring is efficient (BFS); 3-coloring has no known efficient algorithm; the four color theorem
- reductions: scheduling and frequency allocation as graph coloring
- verifying vs. finding a solution; the classes P and NP; the P vs. NP question
- Eulerian (P) vs. Hamiltonian (NP) paths; maximum parsimony
- heuristics, approximations, and greedy coloring